# 01 Plan reading: a floor-plan image becomes rooms

Reads a plan image into the room JSON the app imports ("Import plan" in the Rooms tab, format in `shared/planImport.js`).
Walls are dark lines on a light background; door gaps up to about 0.9 m are sealed; every closed region is a room; each room becomes a rectangle in metres.

This is the classical computer-vision route (OpenCV). SAM 2.1 and Grounding DINO are already cached on this server and can refine or label the regions in the next step. Rooms that are not rectangles (L-shapes) are dropped for now and reported.

Without an image it draws a synthetic test plan, so the cells can be checked end to end. Real plans: upload a PNG next to this notebook and set `IMG_PATH`, and set `PLAN_WIDTH_M` to the real width of the plan.

In [ ]:
import json, numpy as np, cv2
from pathlib import Path
from PIL import Image, ImageDraw
OUT = Path('/workspace/usable-space/out'); OUT.mkdir(parents=True, exist_ok=True)
IMG_PATH = None        # e.g. '/workspace/usable-space/my_plan.png'; None = draw the synthetic test plan
PLAN_WIDTH_M = 9.0     # real width of the plan, wall to wall, in metres
print('opencv', cv2.__version__)

In [ ]:
PX = 100  # pixels per metre of the synthetic plan
ROOMS = {'living': (0, 0, 4.5, 3.6), 'bed1': (4.5, 0, 4.5, 3.6), 'bed2': (0, 3.6, 3.6, 3.4), 'bath': (3.6, 3.6, 2.0, 3.4), 'study': (5.6, 3.6, 3.4, 3.4)}
DOORS = [('v', 4.5, 1.4), ('h', 3.6, 1.2), ('h', 3.6, 4.7), ('v', 5.6, 4.8), ('h', 3.6, 7.0)]  # (direction, line coordinate, start of a 0.9 m gap)
def synthetic_plan():
    pad, T = 60, 8
    im = Image.new('L', (int(9 * PX) + 2 * pad, int(7 * PX) + 2 * pad), 255); d = ImageDraw.Draw(im)
    def line(x0, y0, x1, y1):
        d.rectangle([pad + x0 * PX - T // 2, pad + y0 * PX - T // 2, pad + x1 * PX + T // 2, pad + y1 * PX + T // 2], fill=0)
    for (x, y, w, h) in ROOMS.values():
        line(x, y, x + w, y); line(x, y + h, x + w, y + h); line(x, y, x, y + h); line(x + w, y, x + w, y + h)
    for kind, c, s in DOORS:   # open a door gap by painting it white again
        if kind == 'v': d.rectangle([pad + c * PX - T, pad + s * PX, pad + c * PX + T, pad + (s + 0.9) * PX], fill=255)
        else: d.rectangle([pad + s * PX, pad + c * PX - T, pad + (s + 0.9) * PX, pad + c * PX + T], fill=255)
    return np.asarray(im)
gray = synthetic_plan() if IMG_PATH is None else cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)
Image.fromarray(gray).save(OUT / 'plan_input.png')
print(gray.shape)

In [ ]:
def read_rooms(gray, plan_width_m, door_m=0.9, min_m2=1.5, min_rect=0.75):
    walls = (gray < 128).astype(np.uint8)
    ys, xs = np.where(walls)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    scale = plan_width_m / (x1 - x0 + 1)                      # metres per pixel
    dt = cv2.distanceTransform(walls, cv2.DIST_L2, 3)
    wall_px = max(2.0, 2 * float(np.percentile(dt[walls > 0], 90)))   # wall thickness in pixels
    r = int(round(door_m / scale / 2)) + 1                     # seal gaps up to one door width
    sealed = cv2.dilate(walls, cv2.getStructuringElement(cv2.MORPH_RECT, (2 * r + 1, 2 * r + 1)))
    free = (sealed == 0).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(free, connectivity=4)
    rooms, dropped = [], []
    grow = r + wall_px / 2                                      # undo the sealing and reach the middle of the wall
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if x == 0 or y == 0 or x + w >= gray.shape[1] or y + h >= gray.shape[0]:
            continue                                            # touches the border: the outside
        bx0, by0, bx1, by1 = x - grow, y - grow, x + w + grow, y + h + grow
        rect_fill = area / float(w * h)
        gross = (bx1 - bx0) * (by1 - by0) * scale * scale
        if gross < min_m2: dropped.append(('too small', round(gross, 1))); continue
        if rect_fill < min_rect: dropped.append(('not a rectangle', round(gross, 1), round(rect_fill, 2))); continue
        rooms.append({'x': round((bx0 - x0) * scale, 2), 'y': round((by0 - y0) * scale, 2), 'w': round((bx1 - bx0) * scale, 2), 'd': round((by1 - by0) * scale, 2)})
    rooms.sort(key=lambda r: (round(r['y']), r['x']))
    for k, r in enumerate(rooms): r['id'] = f'r{k + 1}'
    return {'version': 1, 'source': 'opencv wall segmentation', 'footprint': {'w': round((x1 - x0 + 1) * scale, 2), 'd': round((y1 - y0 + 1) * scale, 2)}, 'rooms': rooms}, dropped, scale, (x0, y0)
plan, dropped, scale, origin = read_rooms(gray, PLAN_WIDTH_M)
print(json.dumps(plan, indent=1)); print('dropped:', dropped)

In [ ]:
vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
for r in plan['rooms']:
    p0 = (int(origin[0] + r['x'] / scale), int(origin[1] + r['y'] / scale)); p1 = (int(origin[0] + (r['x'] + r['w']) / scale), int(origin[1] + (r['y'] + r['d']) / scale))
    cv2.rectangle(vis, p0, p1, (40, 80, 230), 3); cv2.putText(vis, r['id'], (p0[0] + 8, p0[1] + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (40, 80, 230), 2)
cv2.imwrite(str(OUT / 'plan_overlay.png'), vis)
(OUT / 'plan_read.json').write_text(json.dumps(plan, indent=1), encoding='utf-8')
print('written', OUT / 'plan_read.json', 'and plan_overlay.png')